<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/Accuracy/accuracy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import densenet121
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
import numpy as np

class DenseNetDogrulukDegerlendirici:
    """
    DenseNet modellerini accuracy metriği kullanarak değerlendiren
    profesyonel ve yapılandırılmış sınıf implementasyonu
    """

    def __init__(self, sinif_sayisi=10, on_egitimli=True):
        self.cihaz = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.sinif_sayisi = sinif_sayisi

        # DenseNet modelini başlat
        self.model = densenet121(pretrained=on_egitimli)
        self.model.classifier = nn.Linear(self.model.classifier.in_features, sinif_sayisi)
        self.model.to(self.cihaz)

        # Kayıp fonksiyonu ve optimizasyon algoritması
        self.kayip_fonksiyonu = nn.CrossEntropyLoss()
        self.optimizasyon = optim.Adam(self.model.parameters(), lr=0.001)

    def dogruluk_hesapla(self, tahminler, gercek_etiketler):
        """
        Zarif ve verimli accuracy hesaplama implementasyonu

        Args:
            tahminler: Model çıktısı tensor'ı
            gercek_etiketler: Gerçek etiket tensor'ı

        Returns:
            float: Doğruluk değeri (0-1 arası)
        """
        # Tahminleri sınıf tahminlerine dönüştür
        tahmin_edilen_siniflar = torch.argmax(tahminler, dim=1)

        # Doğru tahmin sayısını hesapla
        dogru_tahminler = (tahmin_edilen_siniflar == gercek_etiketler).sum().item()

        # Toplam örnek sayısı
        toplam_ornekler = gercek_etiketler.size(0)

        # Doğruluk oranını döndür
        return dogru_tahminler / toplam_ornekler

    def model_degerlendir(self, veri_yukleyici, faz="dogrulama"):
        """
        Kapsamlı model değerlendirmesi ve accuracy metrik analizi

        Args:
            veri_yukleyici: Değerlendirme için DataLoader
            faz: Değerlendirme fazı tanımlayıcısı

        Returns:
            dict: Değerlendirme sonuçlarını içeren sözlük
        """
        self.model.eval()
        toplam_kayip = 0.0
        toplam_dogru = 0
        toplam_ornekler = 0
        tum_tahminler = []
        tum_etiketler = []

        with torch.no_grad():
            for batch_indeksi, (girdiler, etiketler) in enumerate(veri_yukleyici):
                # Veriyi cihaza transfer et
                girdiler, etiketler = girdiler.to(self.cihaz), etiketler.to(self.cihaz)

                # İleri geçiş
                ciktilar = self.model(girdiler)
                kayip = self.kayip_fonksiyonu(ciktilar, etiketler)

                # Bu batch için doğruluk hesapla
                batch_dogruluk = self.dogruluk_hesapla(ciktilar, etiketler)

                # İstatistikleri biriktir
                toplam_kayip += kayip.item()
                batch_dogru = (torch.argmax(ciktilar, dim=1) == etiketler).sum().item()
                toplam_dogru += batch_dogru
                toplam_ornekler += etiketler.size(0)

                # İleri analiz için tahmin ve etiketleri sakla
                tum_tahminler.extend(torch.argmax(ciktilar, dim=1).cpu().numpy())
                tum_etiketler.extend(etiketler.cpu().numpy())

                # Her 50 batch'te progress raporu
                if batch_indeksi % 50 == 0:
                    print(f"Batch {batch_indeksi}/{len(veri_yukleyici)} - "
                          f"Kayıp: {kayip.item():.4f}, "
                          f"Batch Doğruluk: {batch_dogruluk:.4f}")

        # Final metrikleri hesapla
        genel_dogruluk = toplam_dogru / toplam_ornekler
        ortalama_kayip = toplam_kayip / len(veri_yukleyici)

        # Sklearn ile doğrulama
        sklearn_dogruluk = accuracy_score(tum_etiketler, tum_tahminler)

        sonuclar = {
            'faz': faz,
            'dogruluk': genel_dogruluk,
            'sklearn_dogruluk': sklearn_dogruluk,
            'ortalama_kayip': ortalama_kayip,
            'toplam_ornekler': toplam_ornekler,
            'dogru_tahminler': toplam_dogru,
            'dogruluk_yuzde': genel_dogruluk * 100
        }

        return sonuclar

    def dogruluk_izlemeli_egitim(self, egitim_yukleyici, dogrulama_yukleyici, epoch_sayisi=10):
        """
        Gerçek zamanlı accuracy izleme ile eğitim döngüsü

        Args:
            egitim_yukleyici: Eğitim için DataLoader
            dogrulama_yukleyici: Doğrulama için DataLoader
            epoch_sayisi: Eğitim epoch sayısı
        """
        egitim_gecmisi = {
            'egitim_dogruluk': [],
            'dogrulama_dogruluk': [],
            'egitim_kayip': [],
            'dogrulama_kayip': []
        }

        for epoch in range(epoch_sayisi):
            print(f"\nEpoch {epoch+1}/{epoch_sayisi}")
            print("-" * 60)

            # Eğitim fazı
            self.model.train()
            egitim_kayip = 0.0
            egitim_dogru = 0
            egitim_ornekler = 0

            for girdiler, etiketler in egitim_yukleyici:
                girdiler, etiketler = girdiler.to(self.cihaz), etiketler.to(self.cihaz)

                # Gradyanları sıfırla
                self.optimizasyon.zero_grad()

                # İleri geçiş
                ciktilar = self.model(girdiler)
                kayip = self.kayip_fonksiyonu(ciktilar, etiketler)

                # Geri geçiş
                kayip.backward()
                self.optimizasyon.step()

                # İstatistikleri güncelle
                egitim_kayip += kayip.item()
                egitim_dogru += (torch.argmax(ciktilar, dim=1) == etiketler).sum().item()
                egitim_ornekler += etiketler.size(0)

            # Eğitim doğruluğunu hesapla
            egitim_dogruluk = egitim_dogru / egitim_ornekler
            ortalama_egitim_kayip = egitim_kayip / len(egitim_yukleyici)

            # Doğrulama fazı
            dogrulama_sonuclari = self.model_degerlendir(dogrulama_yukleyici, "doğrulama")

            # Geçmişi güncelle
            egitim_gecmisi['egitim_dogruluk'].append(egitim_dogruluk)
            egitim_gecmisi['dogrulama_dogruluk'].append(dogrulama_sonuclari['dogruluk'])
            egitim_gecmisi['egitim_kayip'].append(ortalama_egitim_kayip)
            egitim_gecmisi['dogrulama_kayip'].append(dogrulama_sonuclari['ortalama_kayip'])

            # Epoch sonuçlarını yazdır
            print(f"Eğitim    - Kayıp: {ortalama_egitim_kayip:.4f}, "
                  f"Doğruluk: {egitim_dogruluk:.4f} (%{egitim_dogruluk*100:.2f})")
            print(f"Doğrulama - Kayıp: {dogrulama_sonuclari['ortalama_kayip']:.4f}, "
                  f"Doğruluk: {dogrulama_sonuclari['dogruluk']:.4f} "
                  f"(%{dogrulama_sonuclari['dogruluk_yuzde']:.2f})")

            # En iyi modeli kaydet kontrolü
            if epoch == 0 or dogrulama_sonuclari['dogruluk'] > max(egitim_gecmisi['dogrulama_dogruluk'][:-1]):
                print("🎯 Yeni en iyi doğruluk değeri elde edildi!")

        return egitim_gecmisi

    def detayli_analiz_raporu(self, sonuclar):
        """
        Accuracy sonuçları için detaylı analiz raporu

        Args:
            sonuclar: model_degerlendir() fonksiyonundan dönen sonuçlar
        """
        print("\n" + "="*70)
        print("🔍 DETAYlı ACCURACY ANALİZ RAPORU")
        print("="*70)

        print(f"📊 Genel Performans Metrikleri:")
        print(f"   • Toplam Örnek Sayısı: {sonuclar['toplam_ornekler']:,}")
        print(f"   • Doğru Tahmin Sayısı: {sonuclar['dogru_tahminler']:,}")
        print(f"   • Yanlış Tahmin Sayısı: {sonuclar['toplam_ornekler'] - sonuclar['dogru_tahminler']:,}")

        print(f"\n🎯 Doğruluk Metrikleri:")
        print(f"   • PyTorch Accuracy: {sonuclar['dogruluk']:.6f}")
        print(f"   • Sklearn Accuracy: {sonuclar['sklearn_dogruluk']:.6f}")
        print(f"   • Yüzde Doğruluk: %{sonuclar['dogruluk_yuzde']:.2f}")

        # Performans kategorisi belirleme
        if sonuclar['dogruluk'] >= 0.95:
            kategori = "🌟 Mükemmel"
        elif sonuclar['dogruluk'] >= 0.90:
            kategori = "🎖️ Çok İyi"
        elif sonuclar['dogruluk'] >= 0.80:
            kategori = "👍 İyi"
        elif sonuclar['dogruluk'] >= 0.70:
            kategori = "⚠️ Orta"
        else:
            kategori = "❌ Düşük"

        print(f"\n📈 Performans Kategorisi: {kategori}")
        print(f"💡 Ortalama Kayıp: {sonuclar['ortalama_kayip']:.4f}")

# Profesyonel kullanım örneği
def ana_program():
    """
    DenseNet ile accuracy metriği kullanımının profesyonel demonstrasyonu
    """
    # DenseNet için optimal veri dönüşümleri
    donusum = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    # Değerlendirici sınıfını başlat
    degerlendirici = DenseNetDogrulukDegerlendirici(sinif_sayisi=10)

    # Başlatma bilgileri
    print("🚀 DenseNet Accuracy Değerlendirme Sistemi")
    print("="*50)
    print(f"🖥️  Kullanılan Cihaz: {degerlendirici.cihaz}")
    print(f"🧠 Model Parametreleri: {sum(p.numel() for p in degerlendirici.model.parameters()):,}")
    print(f"🎯 Sınıf Sayısı: {degerlendirici.sinif_sayisi}")

    # Simülasyon verisi ile örnek
    orneklemli_tahminler = torch.randn(100, 10)  # 100 örnek, 10 sınıf
    orneklemli_etiketler = torch.randint(0, 10, (100,))  # Rastgele etiketler

    dogruluk = degerlendirici.dogruluk_hesapla(orneklemli_tahminler, orneklemli_etiketler)
    print(f"\n📊 Örnek Doğruluk Hesaplaması: {dogruluk:.4f} (%{dogruluk*100:.2f})")

    # Tahmin dağılımı analizi
    tahmin_edilen_siniflar = torch.argmax(orneklemli_tahminler, dim=1)
    benzersiz_siniflar, sayimlar = torch.unique(tahmin_edilen_siniflar, return_counts=True)

    print(f"\n📈 Sınıf Tahmin Dağılımı:")
    for sinif, sayim in zip(benzersiz_siniflar, sayimlar):
        print(f"   Sınıf {sinif}: {sayim} tahmin (%{sayim/100*100:.1f})")

    # Simüle edilmiş değerlendirme sonuçları
    simulasyon_sonuclari = {
        'faz': 'test',
        'dogruluk': dogruluk,
        'sklearn_dogruluk': dogruluk,
        'ortalama_kayip': 0.234,
        'toplam_ornekler': 100,
        'dogru_tahminler': int(dogruluk * 100),
        'dogruluk_yuzde': dogruluk * 100
    }

    # Detaylı rapor oluştur
    degerlendirici.detayli_analiz_raporu(simulasyon_sonuclari)

if __name__ == "__main__":
    ana_program()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 33.1MB/s]


🚀 DenseNet Accuracy Değerlendirme Sistemi
🖥️  Kullanılan Cihaz: cpu
🧠 Model Parametreleri: 6,964,106
🎯 Sınıf Sayısı: 10

📊 Örnek Doğruluk Hesaplaması: 0.0900 (%9.00)

📈 Sınıf Tahmin Dağılımı:
   Sınıf 0: 10 tahmin (%10.0)
   Sınıf 1: 7 tahmin (%7.0)
   Sınıf 2: 9 tahmin (%9.0)
   Sınıf 3: 13 tahmin (%13.0)
   Sınıf 4: 13 tahmin (%13.0)
   Sınıf 5: 8 tahmin (%8.0)
   Sınıf 6: 12 tahmin (%12.0)
   Sınıf 7: 13 tahmin (%13.0)
   Sınıf 8: 6 tahmin (%6.0)
   Sınıf 9: 9 tahmin (%9.0)

🔍 DETAYlı ACCURACY ANALİZ RAPORU
📊 Genel Performans Metrikleri:
   • Toplam Örnek Sayısı: 100
   • Doğru Tahmin Sayısı: 9
   • Yanlış Tahmin Sayısı: 91

🎯 Doğruluk Metrikleri:
   • PyTorch Accuracy: 0.090000
   • Sklearn Accuracy: 0.090000
   • Yüzde Doğruluk: %9.00

📈 Performans Kategorisi: ❌ Düşük
💡 Ortalama Kayıp: 0.2340
